# 🚁 Drone vs Bird Detection: Full Pipeline
**Computer Vision | Group Project | IE University MBDS-SEP25**

This unified notebook merges all four milestone notebooks into a single, end-to-end, run-all pipeline:

| Section | Scope |
|---|---|
| **Part 1 · M1+M2: Data Sourcing & Annotation** | Roboflow download, annotation verification, class-balance analysis, train/val/test split |
| **Part 2 · M3: Detector Training** | YOLOv11s & YOLOv11n transfer learning, hyperparameters, training curves |
| **Part 3 · M4: Evaluation & Failure Analysis** | Test-set metrics, confusion matrix, failure gallery, error-by-size |
| **Part 4 · M4+M5: Tracking, Demo & Reproducibility** | ByteTrack object tracking, SAHI small-object inference, live-demo path, reproducibility checklist |

> **How to run:** Mount Google Drive (or set local paths in the Paths cells), then *Runtime → Run all*. Each Part is self-contained but shares Drive paths, so run sequentially on a fresh runtime for a fully reproducible pipeline.

---
# 📦 Part 1 · M1+M2: Data Sourcing & Annotation (drone vs bird)

**Scope:** Dataset acquisition, annotation verification, and train/val/test split

This section downloads the Roboflow Universe "drone-vs-bird" dataset (v3, YOLOv11 format), verifies annotation quality through visual inspection and class-balance analysis, checks for train/val/test leakage, and reorganises the export into the project's folder layout on Google Drive.

**Why Roboflow and not a custom collection?**
The dataset ships with YOLO-format annotations for two classes (`bird` and `drone`) covering a range of backgrounds, distances, and lighting conditions. Collecting and annotating comparable imagery from scratch would take weeks; Roboflow provides a vetted, ready-to-use baseline that meets the assignment requirement of a prepared and annotated image dataset.

**Why verify annotations manually?**
Pre-annotated labels are not always accurate. Visual inspection catches systematic labelling errors (swapped class ids, boxes that miss the object, duplicate boxes) that would otherwise silently degrade training.

## ⚙️ 0 · Environment Setup & Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---- 0. Install deps ----
!pip install -q roboflow "ultralytics>=8.3.0,<9.0.0" "PyYAML>=6.0" "matplotlib>=3.7.0" "pandas>=2.0" "seaborn>=0.12"

import os, glob, random, collections
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import yaml
from IPython.display import Image as IPImage, display

import ultralytics
ultralytics.checks()
print(f"Ultralytics version: {ultralytics.__version__}")

# ---- Set working directory to Drive ----
TARGET_DIR = "/content/drive/MyDrive/!CVIS"
os.makedirs(TARGET_DIR, exist_ok=True)
os.chdir(TARGET_DIR)
print(f"\n Changed working directory to {TARGET_DIR}\n")

# ---- 1. Download dataset from Roboflow ----
from roboflow import Roboflow

# Priority: Colab Secrets > env var > raise clear error
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")

if not ROBOFLOW_API_KEY:
    raise ValueError(
        "ROBOFLOW_API_KEY not set!\n"
        "In Colab: click the Secrets icon (left sidebar) -> add ROBOFLOW_API_KEY.\n"
        "Or paste your key directly here: ROBOFLOW_API_KEY = 'your_key_here'"
    )
print("Roboflow API key loaded OK.")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("avionicsdata").project("drone-vs-bird-object-detection")
dataset = project.version(3).download("yolov11")

ROBOFLOW_DATA_DIR = dataset.location   # original Roboflow export (train/valid/test)
print(f"\nDataset downloaded to: {ROBOFLOW_DATA_DIR}\n")

# ---- 2. Read class info ----
with open(os.path.join(ROBOFLOW_DATA_DIR, "data.yaml")) as f:
    data_yaml = yaml.safe_load(f)

CLASS_NAMES = data_yaml["names"]
print(f"Classes: {dict(enumerate(CLASS_NAMES))}\n")

SPLITS = [s for s in ["train", "valid", "test"]
          if os.path.isdir(os.path.join(ROBOFLOW_DATA_DIR, s, "images"))]
print(f"Splits found: {SPLITS}\n")


def list_images(split):
    p = os.path.join(ROBOFLOW_DATA_DIR, split, "images")
    return sorted(glob.glob(os.path.join(p, "*.jpg")) +
                  glob.glob(os.path.join(p, "*.png")))

def label_path_for(img_path):
    base = os.path.splitext(os.path.basename(img_path))[0]
    split_dir = os.path.dirname(os.path.dirname(img_path))
    return os.path.join(split_dir, "labels", base + ".txt")

def read_boxes(label_file):
    if not os.path.exists(label_file):
        return []
    rows = []
    with open(label_file) as f:
        for line in f:
            parts = line.split()
            if len(parts) == 5:
                cid = int(parts[0])
                xc, yc, w, h = map(float, parts[1:])
                rows.append((cid, xc, yc, w, h))
    return rows


# ---- 3. VISUAL VERIFY ----
print("="*60)
print("STEP 3 - VISUAL VERIFY")
print("="*60)
train_imgs = list_images("train")
sample = random.sample(train_imgs, min(16, len(train_imgs)))
colors = ["red", "lime", "cyan", "yellow", "magenta"]
fig, axes = plt.subplots(4, 4, figsize=(16, 16))
for ax, img_path in zip(axes.flat, sample):
    img = Image.open(img_path)
    W, H = img.size
    ax.imshow(img)
    for cid, xc, yc, w, h in read_boxes(label_path_for(img_path)):
        x, y = (xc - w/2)*W, (yc - h/2)*H
        rect = patches.Rectangle((x, y), w*W, h*H, linewidth=2,
                                  edgecolor=colors[cid % len(colors)], facecolor="none")
        ax.add_patch(rect)
        ax.text(x, max(y-5, 0), CLASS_NAMES[cid],
                color=colors[cid % len(colors)], fontsize=10,
                weight="bold", backgroundcolor="black")
    ax.axis("off")
for ax in axes.flat[len(sample):]:
    ax.axis("off")
plt.suptitle("Annotation check", fontsize=14)
plt.tight_layout()
plt.show()
print("SANITY CHECK | Drone label on a drone? Bird on a bird?")


# ---- 4 & 5. CLASS BALANCE + SPLIT SANITY ----
print("="*60)
print("STEP 4 & 5 - CLASS BALANCE + SPLIT SANITY")
print("="*60)
instances = {s: collections.Counter() for s in SPLITS}
img_counts, images_with_no_label, filenames_per_split = {}, {}, {}
for s in SPLITS:
    imgs = list_images(s)
    img_counts[s] = len(imgs)
    filenames_per_split[s] = set(os.path.basename(p) for p in imgs)
    images_with_no_label[s] = 0
    for img_path in imgs:
        boxes = read_boxes(label_path_for(img_path))
        if not boxes:
            images_with_no_label[s] += 1
        for cid, *_ in boxes:
            instances[s][cid] += 1
total_imgs = sum(img_counts.values())
print("\nImages per split:")
for s in SPLITS:
    pct = 100 * img_counts[s] / total_imgs if total_imgs else 0
    print(f"  {s:6s}: {img_counts[s]:5d} images  ({pct:.0f}%)  | {images_with_no_label[s]} with NO label file")
print(f"  TOTAL : {total_imgs} images")
grand = collections.Counter()
for s in SPLITS:
    grand.update(instances[s])
total_boxes = sum(grand.values())
print("\nClass balance:")
for i, name in enumerate(CLASS_NAMES):
    share = 100 * grand[i] / total_boxes if total_boxes else 0
    print(f"  {name:8s}: {grand[i]:6d} boxes  ({share:.1f}%)")
print("\nLeakage check:")
leak_found = False
for i, a in enumerate(SPLITS):
    for b in SPLITS[i+1:]:
        overlap = filenames_per_split[a] & filenames_per_split[b]
        if overlap:
            leak_found = True
            print(f"  FAIL: {len(overlap)} images shared between {a} and {b}")
if not leak_found:
    print("  OK - No leakage, every image lives in exactly one split.")

# ---- 6. DISTRIBUTION PLOT ----
fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(CLASS_NAMES))
bottom = [0] * len(CLASS_NAMES)
for s in SPLITS:
    vals = [instances[s][i] for i in range(len(CLASS_NAMES))]
    ax.bar(x, vals, bottom=bottom, label=s)
    bottom = [bottom[i] + vals[i] for i in range(len(CLASS_NAMES))]
ax.set_xticks(list(x))
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel("number of boxes (instances)")
ax.set_title("Class distribution by split")
ax.legend(title="split")
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=120)
plt.show()
print("\nDone. 'class_distribution.png' saved.")

In [ ]:
import os, shutil, yaml

TARGET = "/content/drive/MyDrive/!CVIS/data/processed"
SPLIT_RENAME = {"train": "train", "valid": "val", "test": "test"}

os.makedirs(TARGET, exist_ok=True)
for src_split, dst_split in SPLIT_RENAME.items():
    src = os.path.join(ROBOFLOW_DATA_DIR, src_split)   # ROBOFLOW_DATA_DIR set in cell above
    if not os.path.isdir(src):
        print(f"  skip: {src_split} not found")
        continue
    for sub in ["images", "labels"]:
        s = os.path.join(src, sub)
        d = os.path.join(TARGET, dst_split, sub)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
            print(f"  {src_split}/{sub:6s} -> {dst_split}/{sub}  ({len(os.listdir(d))} files)")

data_yaml_out = {
    "path": TARGET,
    "train": "train/images",
    "val":   "val/images",
    "test":  "test/images",
    "nc":    len(CLASS_NAMES),
    "names": CLASS_NAMES,
}
with open(os.path.join(TARGET, "data.yaml"), "w") as f:
    yaml.safe_dump(data_yaml_out, f, sort_keys=False)

print(f"\nDone. Cleaned dataset at: {TARGET}")
print("   Layout: train|val|test each with images/ + labels/")
print("   data.yaml written with 'val' (not 'valid').")

---
# 🚁 Part 2 · M3: Drone vs Bird Detector Training

**Scope:** Transfer-learned object detector

This section fine-tunes a YOLOv11 detector to classify **drone vs bird** using transfer learning from COCO-pretrained weights. We train two model scales, YOLOv11s and YOLOv11n, using identical hyperparameters to compare accuracy against inference speed.

**Why transfer learning?**
YOLOv11 was pretrained on COCO (80 classes, ~118,000 images). Its backbone already encodes edges, textures, and the shapes of moving objects. Fine-tuning only the detection head for our two classes is far more practical than training from scratch, which would overfit badly on 1,950 images.

**Why YOLOv11 and not an older version?**
YOLOv11 (Ultralytics, 2024) improves on YOLOv8 in small-object detection, which matters here since drones and birds at a distance are often smaller than 32×32 px.

## ⚙️ 0 · Path Configuration for Training

In [ ]:
import os, shutil
import yaml as _yaml

# --- Global path constants (shared by Part 2, 3, and 4) ---
DRIVE_ROOT = "/content/drive/MyDrive/!CVIS"
DATA_DIR   = f"{DRIVE_ROOT}/data/processed"
DATA_YAML  = f"{DATA_DIR}/data.yaml"
MODELS_DIR = f"{DRIVE_ROOT}/models"
RUNS_DIR   = f"{DRIVE_ROOT}/runs"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

assert os.path.isdir(DATA_DIR), (
    f"Dataset directory not found: {DATA_DIR}\n"
    "The !CVIS folder is shared by a teammate. In Google Drive, open Shared with me,\n"
    "right-click !CVIS, choose Add shortcut to Drive, place it in My Drive, then\n"
    "Runtime > Disconnect and delete runtime and re-run from the top."
)
assert os.path.exists(DATA_YAML), f"data.yaml not found at {DATA_YAML}"

with open(DATA_YAML) as _f:
    _d = _yaml.safe_load(_f)
NAMES    = _d["names"]
BIRD_ID  = NAMES.index("bird")  if "bird"  in NAMES else 0
DRONE_ID = NAMES.index("drone") if "drone" in NAMES else 1

print(f"DRIVE_ROOT : {DRIVE_ROOT}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"MODELS_DIR : {MODELS_DIR}")
print(f"RUNS_DIR   : {RUNS_DIR}")
print(f"Classes    : {NAMES}  (bird={BIRD_ID}, drone={DRONE_ID})")

for split in ["train", "val", "test"]:
    d = os.path.join(DATA_DIR, split, "images")
    if os.path.isdir(d):
        n = len([f for f in os.listdir(d) if f.lower().endswith((".jpg",".png"))])
        print(f"  {split:5s}: {n} images")

## 🛠️ 1 · Training Configuration & Hyperparameter Rationale

All hyperparameters are declared explicitly, as required by the assignment.

| Parameter | Value | Rationale |
|---|---|---|
| `epochs` | 50 | Enough to fine-tune a pretrained backbone on ~2,000 images without overfitting |
| `batch` | 16 | Fits Colab T4 VRAM (15 GB) at imgsz=640; larger batches stabilise gradient updates |
| `imgsz` | 640 | YOLO standard; 1,280 helps small objects but roughly doubles VRAM usage |
| `optimizer` | AdamW | Adam with weight decay; consistently outperforms SGD for fine-tuning tasks |
| `lr0` | 0.001 | Initial learning rate, lower than the from-scratch default (0.01) since the backbone is already pretrained |
| `lrf` | 0.01 | Final LR = lr0 × lrf = 1e-5, following a cosine decay schedule |
| `momentum` | 0.937 | AdamW beta1; YOLO default, well-validated across detection tasks |
| `weight_decay` | 0.0005 | L2 regularisation to reduce overfitting on the small dataset |
| `warmup_epochs` | 3 | Ramps LR from near-zero to lr0 over the first 3 epochs to avoid early instability |
| `mosaic` | 1.0 | Combines 4 training images into one mosaic to increase contextual variety |
| `copy_paste` | 0.3 | Pastes objects between images; counters the bird-vs-drone imbalance identified in M2 |
| `seed` | 42 | Fixed seed for reproducible runs |

In [ ]:
# Shared hyperparameter dict: changing a value here applies to both model runs.
SHARED_HP = dict(
    data          = DATA_YAML,
    epochs        = 50,
    batch         = 16,
    imgsz         = 640,
    optimizer     = "AdamW",
    lr0           = 0.001,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 0.0005,
    warmup_epochs = 3,
    mosaic        = 1.0,
    copy_paste    = 0.3,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    fliplr        = 0.5,
    flipud        = 0.0,
    seed          = 42,     # fixed seed for reproducibility across runs
    workers       = 2,      # Colab's DataLoader is unstable with the default of 8
    project       = RUNS_DIR,
    save          = True,
    plots         = True,
    verbose       = True,
)
print("Hyperparameters confirmed:", SHARED_HP)

## 🏆 2 · Train YOLOv11s (Primary Model)

**YOLOv11s** (small): 9.4 M parameters. This is the primary model whose `best.pt` will be shared with the evaluation section (M4) and the tracking section (M5).

In [ ]:
from ultralytics import YOLO

# yolo11s.pt is downloaded automatically from Ultralytics (COCO pretrained)
model_s = YOLO("yolo11s.pt")

results_s = model_s.train(
    name = "drone_bird_v1_s",
    **SHARED_HP
)

print("\n✅ YOLOv11s training complete.")
print(f"   Best weights: {results_s.save_dir}/weights/best.pt")

In [ ]:
# Copy best.pt to the shared Drive location — unblocks M4 (evaluation) and M5 (tracking)
best_s_src = f"{results_s.save_dir}/weights/best.pt"
best_s_dst = f"{MODELS_DIR}/best_s.pt"
assert os.path.exists(best_s_src), f"best.pt not found — did training complete? ({best_s_src})"
shutil.copy(best_s_src, best_s_dst)
print(f"✅ YOLOv11s best.pt saved to Drive: {best_s_dst}")

## ⚡ 3 · Train YOLOv11n (Scale-Comparison Bonus)

**YOLOv11n** (nano): 2.6 M parameters, roughly 3.5x fewer than YOLOv11s. We use identical hyperparameters to keep the comparison fair.

**What we are testing:** whether the accuracy drop from s to n is acceptable given the speed gain. In a real counter-UAV system, latency matters: a detector that is even 0.3 s slower could miss a fast-moving drone.

In [ ]:
model_n = YOLO("yolo11n.pt")

results_n = model_n.train(
    name = "drone_bird_v1_n",
    **SHARED_HP
)

print("\n✅ YOLOv11n training complete.")
print(f"   Best weights: {results_n.save_dir}/weights/best.pt")

In [ ]:
best_n_src = f"{results_n.save_dir}/weights/best.pt"
best_n_dst = f"{MODELS_DIR}/best_n.pt"
assert os.path.exists(best_n_src), f"best.pt not found — did training complete? ({best_n_src})"
shutil.copy(best_n_src, best_n_dst)
print(f"✅ YOLOv11n best.pt saved to Drive: {best_n_dst}")

## 📊 4 · Training Results & Scale Comparison

YOLO auto-generates loss curves, a confusion matrix, and a PR curve during training. We display them and build a side-by-side comparison table (mAP, FPS, model size).

In [ ]:
from IPython.display import Image as IPImage, display

def show_run_plots(run_dir, label):
    print(f"\n{'='*60}\n  {label}\n{'='*60}")
    for fname in ["results.png", "confusion_matrix_normalized.png", "PR_curve.png"]:
        fpath = os.path.join(str(run_dir), fname)
        if os.path.exists(fpath):
            print(f"\n--- {fname} ---")
            display(IPImage(fpath, width=800))
        else:
            print(f"  (not found: {fname})")

show_run_plots(results_s.save_dir, "YOLOv11s: Primary Model")
show_run_plots(results_n.save_dir, "YOLOv11n: Nano (Scale Comparison)")

In [ ]:
import time
import torch
import pandas as pd

def get_val_metrics(results):
    d = results.results_dict
    return {
        "mAP50":     round(d.get("metrics/mAP50(B)",     0), 4),
        "mAP50-95":  round(d.get("metrics/mAP50-95(B)",  0), 4),
        "Precision": round(d.get("metrics/precision(B)", 0), 4),
        "Recall":    round(d.get("metrics/recall(B)",    0), 4),
    }

def measure_fps(model, runs=50, imgsz=640):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dummy  = torch.zeros(1, 3, imgsz, imgsz).to(device)
    for _ in range(5):
        model.predict(source=dummy, verbose=False)
    t0 = time.perf_counter()
    for _ in range(runs):
        model.predict(source=dummy, verbose=False)
    return round(runs / (time.perf_counter() - t0), 1)

m_s   = get_val_metrics(results_s)
m_n   = get_val_metrics(results_n)
fps_s = measure_fps(YOLO(best_s_dst))
fps_n = measure_fps(YOLO(best_n_dst))
sz_s  = round(os.path.getsize(best_s_dst) / 1e6, 1)
sz_n  = round(os.path.getsize(best_n_dst) / 1e6, 1)

scale_df = pd.DataFrame([
    {"Model": "YOLOv11s", **m_s, "FPS": fps_s, "Size(MB)": sz_s},
    {"Model": "YOLOv11n", **m_n, "FPS": fps_n, "Size(MB)": sz_n},
]).set_index("Model")

print("\n===== SCALE COMPARISON (val split) =====")
display(scale_df)
scale_df.to_csv(f"{RUNS_DIR}/val_scale_comparison.csv")

## 🔍 5 · Quick Inference Visual Check

Before handing off `best.pt`, visually confirm:
- Boxes land on actual objects, not background
- Labels correctly show `bird` or `drone`
- Confidence scores look reasonable (above 0.3)

In [ ]:
import glob, random
from IPython.display import Image as IPImage, display

val_imgs = (glob.glob(f"{DATA_DIR}/val/images/*.jpg") +
            glob.glob(f"{DATA_DIR}/val/images/*.png"))
assert len(val_imgs) > 0, f"No validation images at {DATA_DIR}/val/images/"
sample = random.sample(val_imgs, min(6, len(val_imgs)))

model_best = YOLO(best_s_dst)
INFER_OUT  = f"{RUNS_DIR}/inference_check"

preds = model_best.predict(
    source  = sample,
    conf    = 0.25,
    save    = True,
    project = INFER_OUT,
    name    = "val_sample",
    exist_ok= True,
    verbose = False,
)

saved = sorted(glob.glob(f"{INFER_OUT}/val_sample*/*.jpg") +
               glob.glob(f"{INFER_OUT}/val_sample*/*.png"))
for p in saved[:6]:
    display(IPImage(p, width=640))

print("\nCHECK: do boxes match the correct class? (bird=0, drone=1)")

---
# 🔬 Part 3 · M4: Evaluation & Failure Analysis (drone vs bird)

**Scope:** Test-set evaluation, confusion matrix, failure-case gallery, error-by-size analysis

This section evaluates the YOLOv11s detector trained in Part 2 on the held-out test split (278 images, 302 objects). We report the four standard detection metrics (mAP50, mAP50-95, precision, and recall) and go beyond the numbers with a confusion matrix, a categorised failure gallery, and a size-stratified error analysis.

**Why evaluate on the test set and not the validation set?**
The validation set is used during training to pick the best checkpoint, so its metrics are slightly optimistic: the model has effectively been guided by them through checkpoint selection. The test set is completely held out and untouched until this section, giving an unbiased estimate of real-world performance.

**Why analyse failure cases by category?**
Aggregate metrics can hide the direction of errors. For counter-UAV systems, missing a drone (false negative) is operationally more severe than raising a bird alarm (false positive). Categorised failures make this asymmetry visible.

In [ ]:
import os
import yaml
import shutil

LOCAL_RUNS  = "/content/runs"
RESULTS_DIR = f"{DRIVE_ROOT}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LOCAL_RUNS, exist_ok=True)

BEST_S = best_s_dst
BEST_N = best_n_dst

assert os.path.exists(BEST_S), (
    f"Primary weights not found: {BEST_S}\n"
    "Run Part 2 (M3) above to train and export best_s.pt to the shared Drive first."
)

# Confirm the test split exists
TEST_DIR = os.path.join(DATA_DIR, "test", "images")
assert os.path.isdir(TEST_DIR), f"Test images directory not found: {TEST_DIR}"
n_test = len([f for f in os.listdir(TEST_DIR) if f.endswith((".jpg",".png"))])
print(f"Test images: {n_test}")
print(f"Primary weights: {BEST_S}")
print(f"Results dir: {RESULTS_DIR}")

In [ ]:
import shutil, yaml, glob

USE_LOCAL_STAGE = True

if USE_LOCAL_STAGE:
    LOCAL_DATA = "/content/cvis_eval"
    for sub in ("test/images", "test/labels"):
        s_dir = f"{DATA_DIR}/{sub}"
        d_dir = f"{LOCAL_DATA}/{sub}"
        if not os.path.isdir(d_dir):
            os.makedirs(os.path.dirname(d_dir), exist_ok=True)
            shutil.copytree(s_dir, d_dir)
        else:
            print(f"  already staged: {d_dir}")

    EVAL_YAML = f"{LOCAL_DATA}/data.yaml"
    with open(EVAL_YAML, "w") as f:
        yaml.safe_dump({
            "path": LOCAL_DATA,
            "train": "test/images",
            "val":   "test/images",
            "test":  "test/images",
            "nc": len(NAMES), "names": NAMES
        }, f, sort_keys=False)
    TEST_IMG_DIR = f"{LOCAL_DATA}/test/images"
    TEST_LBL_DIR = f"{LOCAL_DATA}/test/labels"
else:
    EVAL_YAML    = DATA_YAML
    TEST_IMG_DIR = f"{DATA_DIR}/test/images"
    TEST_LBL_DIR = f"{DATA_DIR}/test/labels"

test_imgs = sorted(
    glob.glob(f"{TEST_IMG_DIR}/*.jpg") + glob.glob(f"{TEST_IMG_DIR}/*.png")
)
print(f"Staged {len(test_imgs)} test images for evaluation.")

## 1 · Test-Set Metrics

We run the detector on the **test** split and report four metrics:

- **Precision**: of the boxes called *drone*, how many really were. Low = false alarms on birds.
- **Recall**: of the real drones, how many we caught. Low = missed threats (the costly failure for counter-UAV).
- **mAP50**: detection quality at IoU ≥ 0.50 (found it and roughly localised it).
- **mAP50-95**: same, averaged over stricter IoU thresholds, so it measures how *tight* the boxes are.

In [ ]:
from ultralytics import YOLO

model_s = YOLO(BEST_S)

metrics_s = model_s.val(
    data    = EVAL_YAML,
    split   = "test",
    project = LOCAL_RUNS,
    name    = "eval_test_s",
    plots   = True,
    conf    = 0.001,      # low threshold so the PR curve / mAP integrate the full curve
    iou     = 0.6,        # NMS IoU
    verbose = True,
)
EVAL_S_DIR = metrics_s.save_dir
print("Run folder:", EVAL_S_DIR)

for fig in ["confusion_matrix.png", "confusion_matrix_normalized.png",
            "PR_curve.png", "F1_curve.png", "P_curve.png", "R_curve.png"]:
    fsrc = os.path.join(str(EVAL_S_DIR), fig)
    if os.path.exists(fsrc):
        shutil.copy(fsrc, f"{RESULTS_DIR}/test_s_{fig}")
print("Figures copied to:", RESULTS_DIR)

In [ ]:
import pandas as pd

def overall_row(m, label):
    d = m.results_dict
    return {
        "Model":     label,
        "mAP50":     round(d.get("metrics/mAP50(B)",    0.0), 4),
        "mAP50-95":  round(d.get("metrics/mAP50-95(B)", 0.0), 4),
        "Precision": round(d.get("metrics/precision(B)",0.0), 4),
        "Recall":    round(d.get("metrics/recall(B)",   0.0), 4),
    }

def per_class_table(m, names):
    rows = []
    for i, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        rows.append({
            "Class":     names[c],
            "Precision": round(p, 4),
            "Recall":    round(r, 4),
            "mAP50":     round(ap50, 4),
            "mAP50-95":  round(ap, 4),
        })
    return pd.DataFrame(rows)

overall_s  = pd.DataFrame([overall_row(metrics_s, "YOLOv11s (test)")])
perclass_s = per_class_table(metrics_s, NAMES)

print("OVERALL: test set")
display(overall_s.set_index("Model"))
print("\nPER-CLASS: test set")
display(perclass_s.set_index("Class"))

On 278 unseen test images (302 objects), the detector achieves mAP50 = 0.98, with 0.95 precision and 0.97 recall, confirming that the model generalises well to unseen data with no sign of overfitting. Most importantly, drone recall reaches 1.00, meaning every drone in the test set was detected, with a drone mAP50 of 0.99.

The main limitation is the gap between mAP50 (0.98) and mAP50–95 (0.62), indicating reliable detection but less precise localisation at stricter IoU thresholds. This is particularly evident for drones, which achieve higher mAP50 but lower mAP50–95 than birds (0.59 vs. 0.64), reflecting the challenge of tightly localising small, distant targets where minor box offsets significantly reduce IoU. Birds remain the weaker, minority class across all metrics and account for most residual errors.

## 2 · Confusion Matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# ultralytics >= 8.3: confusion_matrix.matrix; older versions store it differently
try:
    cm_raw = metrics_s.confusion_matrix.matrix
except AttributeError:
    cm_raw = metrics_s.confusion_matrix

cm = cm_raw.astype(int)
labels = NAMES + ["background"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels, cbar=False, ax=axes[0])
axes[0].set_title("Confusion matrix: counts (test set)")
axes[0].set_xlabel("True"); axes[0].set_ylabel("Predicted")

col_sum = cm.sum(axis=0, keepdims=True)
cm_norm = np.divide(cm, col_sum, out=np.zeros_like(cm, dtype=float), where=col_sum != 0)
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=labels, yticklabels=labels, cbar=False, ax=axes[1])
axes[1].set_title("Confusion matrix: normalised (per true class)")
axes[1].set_xlabel("True"); axes[1].set_ylabel("Predicted")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/custom_confusion_matrix.png", dpi=120)
plt.show()

The error direction is more important than the error count. All class confusions go in the safer direction: 4 birds were classified as drones, while no drones were misclassified as birds or missed entirely. For counter-UAV use, this is the preferred failure profile: the system does not let drones pass undetected; its mistakes are additional bird false alarms that an operator can dismiss. The main operational cost is therefore the bird→drone confusion, which may contribute to alarm fatigue, while the empty drone-error row is the key security result.

## 3 · PR & F1 Curves

The PR curve shows how much precision survives as we push recall up; the F1 curve shows which confidence threshold balances them best. We use that threshold to pick the single operating `conf` for the live demo.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

curve_files = {
    "Precision–Recall": ["BoxPR_curve.png", "PR_curve.png"],
    "F1":               ["BoxF1_curve.png", "F1_curve.png"],
    "Precision":        ["BoxP_curve.png",  "P_curve.png"],
    "Recall":           ["BoxR_curve.png",  "R_curve.png"],
}
for label, candidates in curve_files.items():
    for fn in candidates:
        fpath = os.path.join(str(EVAL_S_DIR), fn)
        if os.path.exists(fpath):
            img = mpimg.imread(fpath)
            plt.figure(figsize=(8, 6))
            plt.imshow(img); plt.axis("off"); plt.title(f"{label}  ({fn})")
            plt.show()
            break
    else:
        print(f"(not found: {label} curve)")

## 4 · Failure-Case Gallery

Metrics quantify performance, but they do not reveal the nature of the errors. To go beyond the numbers, we run the detector on the test set at the demo operating threshold (**conf = 0.25**), match predictions to ground truth by IoU, and assign each error to one of four categories:

| **Error category**   | **Definition**                                                                            |
| :------------------- | :---------------------------------------------------------------------------------------- |
| **Bird → Drone**     | A bird is incorrectly classified as a drone (false positive).                             |
| **Drone → Bird**     | A drone is incorrectly classified as a bird (misclassification).                          |
| **Missed detection** | A ground-truth object is not matched by any prediction (false negative).                  |
| **Ghost detection**  | A predicted bounding box does not correspond to any ground-truth object (false positive on background). |

In [ ]:
import glob

CONF_OP   = 0.25   # operating confidence for the gallery (matches the demo default)
MATCH_IOU = 0.50   # IoU threshold to call a prediction a match to a GT box

def load_gt(label_path, W, H):
    # Read a YOLO-format label file -> list of (cls, [x1,y1,x2,y2], area_frac).
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    for line in open(label_path):
        parts = line.split()
        if len(parts) < 5:
            continue
        cls, cx, cy, w, h = int(parts[0]), *map(float, parts[1:5])
        x1, y1 = (cx - w / 2) * W, (cy - h / 2) * H
        x2, y2 = (cx + w / 2) * W, (cy + h / 2) * H
        boxes.append((cls, [x1, y1, x2, y2], w * h))   # w*h (normalised) == area fraction
    return boxes

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter == 0:
        return 0.0
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter)

print(f"CONF_OP={CONF_OP}, MATCH_IOU={MATCH_IOU}")
print("Functions defined. Running inference on test set...")

In [ ]:
import cv2

# Per-object detection records and per-image failure cases (for the gallery).
gt_records   = []   # one row per ground-truth object: {class, area_frac, detected, confused}
failures     = {"bird_as_drone": [], "drone_as_bird": [], "missed": [], "ghost": []}

# Batch inference with stream=True avoids holding all 278 results in memory simultaneously.
for img_path, res in zip(test_imgs, model_s.predict(test_imgs, conf=CONF_OP, iou=0.6,
                                                     verbose=False, stream=True)):
    H, W  = res.orig_shape          # image dimensions from the result object
    stem  = os.path.splitext(os.path.basename(img_path))[0]
    gts   = load_gt(f"{TEST_LBL_DIR}/{stem}.txt", W, H)

    preds = []
    for b in res.boxes:
        preds.append((int(b.cls[0]), float(b.conf[0]), [float(v) for v in b.xyxy[0]]))
    preds.sort(key=lambda p: -p[1])           # greedily match highest-confidence preds first

    matched_pred = [False] * len(preds)
    for gcls, gbox, garea in gts:
        best_iou, best_j = 0.0, -1
        for j, (pcls, pconf, pbox) in enumerate(preds):
            if matched_pred[j]:
                continue
            v = iou(gbox, pbox)
            if v > best_iou:
                best_iou, best_j = v, j
        detected = best_iou >= MATCH_IOU
        confused = False
        if detected:
            matched_pred[best_j] = True
            pcls = preds[best_j][0]
            if pcls != gcls:
                confused = True
                tag = "bird_as_drone" if gcls == BIRD_ID else "drone_as_bird"
                failures[tag].append((img_path, gts, preds))
        else:
            failures["missed"].append((img_path, gts, preds))
        gt_records.append({"class": gcls, "area_frac": garea,
                           "detected": detected, "confused": confused})

    # Ghost: predictions with no GT match
    for j, (pcls, pconf, pbox) in enumerate(preds):
        if not matched_pred[j]:
            failures["ghost"].append((img_path, gts, preds))

print("Failure counts:")
for k, v in failures.items():
    print(f"  {k}: {len(v)}")

In [ ]:
GREEN = (0, 200, 0)     # ground truth
RED   = (0, 0, 230)      # prediction (BGR)

def annotate(img_path, gts, preds):
    im = cv2.imread(img_path)
    for cls, box, _ in gts:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(im, (x1, y1), (x2, y2), GREEN, 2)
        cv2.putText(im, f"GT:{NAMES[cls]}", (x1, max(0, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, GREEN, 2)
    for cls, conf, box in preds:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(im, (x1, y1), (x2, y2), RED, 2)
        cv2.putText(im, f"{NAMES[cls]} {conf:.2f}", (x1, min(im.shape[0] - 5, y2 + 16)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, RED, 2)
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

TITLES = {
    "bird_as_drone": "Bird called DRONE  (false alarm)",
    "drone_as_bird": "Drone called BIRD  (misclassified threat)",
    "missed":        "Missed object  (false negative)",
    "ghost":         "Ghost box on background  (false positive)",
}

def show_gallery(kind, n=4):
    cases = failures[kind][:n]
    if not cases:
        print(f"  No {kind} cases found."); return
    fig, axes = plt.subplots(1, len(cases), figsize=(5 * len(cases), 5))
    if len(cases) == 1:
        axes = [axes]
    fig.suptitle(TITLES[kind], fontsize=13)
    for ax, (img_path, gts, preds) in zip(axes, cases):
        ax.imshow(annotate(img_path, gts, preds))
        ax.axis("off")
        ax.set_title(os.path.basename(img_path), fontsize=8)
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/failure_{kind}.png", dpi=100)
    plt.show()

for kind in ["bird_as_drone", "drone_as_bird", "missed", "ghost"]:
    show_gallery(kind)

At the demo operating threshold (**conf = 0.25**), no drone is misclassified as a bird, while only **5 bird images** are labelled as drones, and most appear to be duplicate detections of the same bird. The dominant error category is **ghost detections**: the **17 ghost detections** identified at **conf = 0.25** do not all correspond to genuine background false positives. Visual inspection reveals three distinct causes.

First, some are **true hallucinations**, where low-confidence detections appear over sky textures or treeline clutter; these are genuine detector errors and could be reduced by increasing the confidence threshold or applying stricter NMS.

Second, some detections correspond to **real but unlabelled objects**, typically birds omitted during annotation. These are counted as ghosts despite representing correct detections, highlighting a limitation in the dataset rather than the model.

Third, some ghosts arise from **duplicate or loosely localised detections**: after matching the highest-IoU prediction to the ground-truth box, remaining overlapping predictions with lower confidence are left unmatched and counted as ghosts. A stricter NMS threshold would suppress most of these.

## 5 · Error by Object Size

Distant (small) targets are the hard case for counter-UAV, so we measure performance as a function of size. Reusing the IoU matching from failure analysis, we bin every ground-truth object by **relative area** (box area ÷ image area, making it comparable across resolutions) and compute, per bin and class, the **detection rate** (recall) and the **confusion rate** (detected but wrong class).

Bins: `tiny <0.1%`, `small 0.1–1%`, `medium 1–5%`, `large >5%`.

In [ ]:
df = pd.DataFrame(gt_records)
df["class_name"] = df["class"].map({BIRD_ID: "bird", DRONE_ID: "drone"})

bins      = [0, 0.001, 0.01, 0.05, 1.0]
labels_sz = ["tiny\n(<0.1%)", "small\n(0.1-1%)", "medium\n(1-5%)", "large\n(>5%)"]
df["size_bin"] = pd.cut(df["area_frac"], bins=bins, labels=labels_sz, include_lowest=True)

# Detection rate (recall) and confusion rate per (size, class).
agg = (df.groupby(["size_bin", "class_name"], observed=True)
         .agg(n=("detected", "size"),
              detection_rate=("detected", "mean"),
              confusion_rate=("confused", "mean"))
         .reset_index())
print("Per size-bin × class:")
display(agg)

agg.to_csv(f"{RESULTS_DIR}/error_by_size.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

piv_det = agg.pivot(index="size_bin", columns="class_name", values="detection_rate")
piv_det.plot(kind="bar", ax=axes[0], color={"bird": "#4C72B0", "drone": "#DD8452"}, rot=0)
axes[0].set_title("Detection rate (recall) by object size")
axes[0].set_ylabel("detection rate"); axes[0].set_ylim(0, 1.05)
axes[0].set_xlabel("object size (relative area)")
axes[0].legend(title="true class")
for c in axes[0].containers:
    axes[0].bar_label(c, fmt="%.2f", fontsize=8, label_type="center")

piv_n = agg.pivot(index="size_bin", columns="class_name", values="n")
piv_n.plot(kind="bar", ax=axes[1], color={"bird": "#4C72B0", "drone": "#DD8452"}, rot=0)
axes[1].set_title("Ground-truth object count by size")
axes[1].set_ylabel("# objects"); axes[1].set_xlabel("object size (relative area)")
axes[1].legend(title="true class")
for c in axes[1].containers:
    axes[1].bar_label(c, fontsize=8, label_type="center")

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/detection_rate_by_size.png", dpi=120)
plt.show()

Detection recall remains near-perfect across object sizes: drone recall is **1.00** in every size bin, while bird recall decreases only for the smallest objects. The main size-dependent effect is therefore **bird→drone misclassification**, which is confined to *tiny* (1/4) and *small* (4/49) birds and disappears for medium and large targets. In practice, the model rarely fails to detect an object but occasionally misclassifies small, low-contrast birds as drones.

Two caveats are worth noting. First, the size distribution is highly imbalanced: **243 of 302 objects** belong to the *small* category, whereas the *tiny* bin contains only **6 objects**, so the lower recall for tiny birds (3/4) should be interpreted cautiously. Second, recall does not reflect localisation quality. The lower **mAP50–95** indicates that bounding-box accuracy also degrades on small objects, even when they are successfully detected.

## 6 · s vs n on the Test Set

Part 2 compared YOLOv11s and YOLOv11n on the validation set, where nano was marginally ahead. Here we extend that comparison to the test set.

In [ ]:
rows = [overall_row(metrics_s, "YOLOv11s (test)")]

if os.path.exists(BEST_N):
    model_n_eval = YOLO(BEST_N)
    metrics_n    = model_n_eval.val(data=EVAL_YAML, split="test", project=LOCAL_RUNS,
                                    name="eval_test_n", plots=False, conf=0.001, iou=0.6, verbose=False)
    rows.append(overall_row(metrics_n, "YOLOv11n (test)"))
    sz_s_mb = round(os.path.getsize(BEST_S) / 1e6, 1)
    sz_n_mb = round(os.path.getsize(BEST_N) / 1e6, 1)
    print(f"Model size: s={sz_s_mb} MB, n={sz_n_mb} MB")
else:
    print("best_n.pt not found. Showing primary model only.")

cmp = pd.DataFrame(rows).set_index("Model")
display(cmp)
cmp.to_csv(f"{RESULTS_DIR}/test_metrics_scale_comparison.csv")

On the test set, **YOLOv11n** and **YOLOv11s** achieve similar detection accuracy: nano reaches **mAP50 = 0.971** and **mAP50–95 = 0.609**, compared with **0.980** and **0.617** for small. The ~0.009 mAP50 gap across 302 test objects is within sampling noise and should not be read as a definitive ranking. However, nano is approximately **3.5× smaller** (5.5 vs. 19.2 MB) and **3× faster** (281 vs. 867 ms/image on CPU), at the cost of lower recall (**0.944** vs. **0.971**).

For the counter-UAV application, this recall advantage is more important than the additional computational cost, as missed drones are operationally more critical than slower inference. Therefore, **YOLOv11s** remains the preferred model, while **YOLOv11n** is a suitable alternative for resource-constrained edge devices where size and speed take priority.

### Evaluation Summary

The proposed detector achieves **mAP50 = 0.98** on unseen data, with **drone recall = 1.00** and no missed or misclassified drones. The remaining errors are conservative: bird false alarms and background ghost detections, which increase operator workload but do not compromise drone detection.

**Limitations.**

* **Localisation accuracy.** Although detection recall is near-perfect, the drop from **mAP50 (0.98)** to **mAP50–95 (0.62)** indicates reduced bounding-box precision, particularly for small, distant objects.
* **Small-object classification.** Residual class confusions are limited to tiny, low-contrast birds misclassified as drones; medium and large birds are rarely confused.
* **Background false positives.** At the operating threshold (**conf = 0.25**), 17 images are flagged as ghost detections, mixing three causes: genuine low-confidence false alarms, real but unlabelled objects, and duplicate boxes.

---
# 🎯 Part 4 · M4+M5: Tracking, Demo & Reproducibility

This section runs locally against the unzipped project folder, no `drive.mount` required if you have the data locally.
Device is auto-detected; a CUDA gate up top fails loudly if the GPU isn’t actually usable.

> **Submission note:** the brief requires a *Colab* notebook. When running in Colab, the paths above (Part 1–3) already point into Drive. The local-dev path cells below use auto-detection so they work in both environments.

## ⚙️ 0 · Environment & GPU Check

Dependencies are managed by your `.venv` (installed via `uv`), so there is **no** additional `pip install`
cell here. If the GPU check below fails, fix the environment before running anything else; do not run on CPU by accident.

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print("device:", torch.cuda.get_device_name(0), "| compute capability:", cap)
    # Real kernel test
    x = torch.randn(2000, 2000, device="cuda")
    _ = (x @ x).sum().item()
    print("✅ GPU matmul executed. Good to go.")
    DEVICE = 0  # YOLO device id
else:
    print("⚠️ Running on CPU — tracking and SAHI will be slower.")
    DEVICE = "cpu"

## 📁 1 · Paths (local / auto-detect)

Set `PROJECT_ROOT` to the folder that contains `data/`, `models/`, `VIDEOS/`.
The cell auto-detects it by searching for `data/processed/data.yaml`; override manually if needed.

In [ ]:
import os, glob
from pathlib import Path

# On Colab with Drive mounted (from Part 1), use Drive path directly.
# Locally: search only immediate parents (no recursive glob on Drive).
DRIVE_ROOT_P4 = "/content/drive/MyDrive/!CVIS"

if os.path.isdir(DRIVE_ROOT_P4):
    PROJECT_ROOT = Path(DRIVE_ROOT_P4)
else:
    def _find_root(start="."):
        here = Path(start).resolve()
        for c in [here] + list(here.parents[:8]):
            if (c / "data" / "processed" / "data.yaml").exists():
                return c
        return None
    PROJECT_ROOT = _find_root(".")
    if PROJECT_ROOT is None:
        raise RuntimeError(
            "Cannot locate project root. Set PROJECT_ROOT manually:\n"
            "  PROJECT_ROOT = Path('/your/path/to/!CVIS')"
        )

print("PROJECT_ROOT:", PROJECT_ROOT)

DATA_DIR_P   = PROJECT_ROOT / "data" / "processed"
DATA_YAML_P  = str(DATA_DIR_P / "data.yaml")
MODELS_DIR_P = PROJECT_ROOT / "models"
RUNS_DIR_P   = str(PROJECT_ROOT / "runs")
VIDEOS_DIR   = PROJECT_ROOT / "VIDEOS"
os.makedirs(RUNS_DIR_P, exist_ok=True)

PRIMARY_WEIGHTS = str(MODELS_DIR_P / "best_s.pt")
if not os.path.exists(PRIMARY_WEIGHTS):
    PRIMARY_WEIGHTS = str(MODELS_DIR_P / "best_n.pt")
    print("WARNING: best_s.pt not found, falling back to best_n.pt")
assert os.path.exists(PRIMARY_WEIGHTS), (
    f"No model weights found in {MODELS_DIR_P}.\n"
    "Run Part 2 (training) first to generate best_s.pt and best_n.pt."
)
print("PRIMARY_WEIGHTS:", PRIMARY_WEIGHTS)

import yaml as _yaml
with open(DATA_YAML_P) as _f:
    _d2 = _yaml.safe_load(_f)
_d2["path"] = str(DATA_DIR_P).replace("\\", "/")
with open(DATA_YAML_P, "w") as _f:
    _yaml.safe_dump(_d2, _f, sort_keys=False)
print("data.yaml path ->", _d2["path"])

## ✅ A · Test-Set Evaluation: REQUIRED (brief item 5)

Training logs *validation* metrics; the brief asks for **test** metrics. This runs the held-out
test split for both model scales, so the comparison is on data neither model saw.

In [ ]:
from ultralytics import YOLO
import pandas as pd
from pathlib import Path

rows = []
for tag in ['n', 's']:
    w = str(MODELS_DIR_P / f'best_{tag}.pt')
    if not os.path.exists(w):
        print(f"Skipping YOLOv11{tag}: weights not found at {w}")
        continue
    m = YOLO(w)
    r = m.val(data=DATA_YAML_P, split='test', device=DEVICE,
              project=RUNS_DIR_P, name=f'test_eval_{tag}', exist_ok=True, verbose=False)
    rows.append({'model': f'YOLOv11{tag}',
                 'mAP50': round(r.box.map50, 4), 'mAP50-95': round(r.box.map, 4),
                 'precision': round(r.box.mp, 4), 'recall': round(r.box.mr, 4)})

test_df = pd.DataFrame(rows)
test_df.to_csv(str(Path(RUNS_DIR_P) / 'test_set_comparison.csv'), index=False)

print('\n' + '='*52)
print('  TEST-SET COMPARISON  (this is the deck slide)')
print('='*52)
print(test_df.to_string(index=False))
print('='*52)
if len(test_df) > 1:
    win = test_df.loc[test_df['mAP50-95'].idxmax(), 'model']
    print(f'Higher mAP50-95: {win}')
try:
    from IPython.display import display
    display(test_df)
except Exception:
    pass

In [ ]:
# Per-class test AP (drone vs bird) for the primary model -> failure-analysis slide.
m_primary = YOLO(PRIMARY_WEIGHTS)
r_primary = m_primary.val(data=DATA_YAML_P, split='test', device=DEVICE,
              project=RUNS_DIR_P, name='test_eval_perclass', exist_ok=True, verbose=False)
for i, name in r_primary.names.items():
    try:    print(f'{name:6} test AP50 = {r_primary.box.ap50[i]:.4f}')
    except Exception: print(f'{name:6} (no test instances)')
print('\nConfusion matrix saved under:', r_primary.save_dir)

## 🎯 B · Object Tracking (committed bonus)

`model.track()` + ByteTrack: persistent IDs across frames in one call, writes an annotated mp4.
Full-resolution clips track comfortably on a modern GPU, no need to trim.

In [ ]:
# List available video clips
if VIDEOS_DIR.exists():
    for p in sorted(glob.glob(str(VIDEOS_DIR / '*.mp4'))):
        print(os.path.basename(p))
else:
    print(f"VIDEOS_DIR not found: {VIDEOS_DIR}")
    print("Set VIDEOS_DIR manually to the folder containing your .mp4 clips.")

In [ ]:
from ultralytics import YOLO

# Update this path to one of the clips printed above
CLIP = str(VIDEOS_DIR / '11498840-hd_1920_1080_30fps.mp4')   # large foreground drone
if not os.path.exists(CLIP):
    mp4s = glob.glob(str(VIDEOS_DIR / '*.mp4'))
    CLIP  = mp4s[0] if mp4s else None
    print(f"Default clip not found — using: {CLIP}")

if CLIP:
    model_track = YOLO(PRIMARY_WEIGHTS)
    # stream=True + consuming the generator avoids the RAM-accumulation warning;
    # save=True still writes the annotated video frame-by-frame.
    gen = model_track.track(
        source=CLIP, tracker='bytetrack.yaml', conf=0.25, iou=0.5,
        persist=True, save=True, device=DEVICE,
        project=RUNS_DIR_P, name='track_demo', exist_ok=True, stream=True, verbose=False)
    for _ in gen:
        pass
    print('Annotated tracking video written under:', str(Path(RUNS_DIR_P) / 'track_demo'))
else:
    print('No .mp4 clips found in VIDEOS_DIR. Skipping tracking.')

In [ ]:
import glob, cv2, os
from pathlib import Path

src_dir = str(Path(RUNS_DIR_P) / "track_demo")
avis = glob.glob(os.path.join(src_dir, "*.avi"))
made = []
for avi in avis:
    mp4 = os.path.splitext(avi)[0] + "_playable.mp4"
    cap = cv2.VideoCapture(avi)
    if not cap.isOpened():
        print(f"Could not open {avi}")
        continue
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = None
    for cc in ("avc1", "mp4v"):   # H.264 preferred; mp4v as fallback
        candidate = cv2.VideoWriter(mp4, cv2.VideoWriter_fourcc(*cc), fps, (w, h))
        if candidate.isOpened():
            writer = candidate
            break
        candidate.release()
    if writer is None:
        print(f"Could not create VideoWriter for {mp4}")
        cap.release()
        continue
    n = 0
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        writer.write(fr)
        n += 1
    cap.release()
    writer.release()
    made.append(mp4)
    print(f"Re-encoded -> {mp4}  ({n} frames)")
if not made:
    print("No .avi files found to re-encode (output may already be .mp4).")

### B.1 · Tracking Stability (rigorous, not a gimmick)

Without ground-truth track IDs, MOTA/IDF1 cannot be computed. Report a defensible proxy:
unique IDs assigned vs. true object count. Excess unique IDs on a single-object clip indicate ID switches.

In [ ]:
import numpy as np

if CLIP and os.path.exists(CLIP):
    model_stab = YOLO(PRIMARY_WEIGHTS)
    ids_seen, per_frame = set(), []
    for r in model_stab.track(source=CLIP, tracker='bytetrack.yaml', persist=True,
                         stream=True, conf=0.25, device=DEVICE, verbose=False):
        if r.boxes is not None and r.boxes.id is not None:
            ids = r.boxes.id.int().tolist()
            ids_seen.update(ids); per_frame.append(len(ids))
    print(f'Frames with detections : {len(per_frame)}')
    print(f'Unique track IDs        : {len(ids_seen)}')
    print(f'Avg objects per frame   : {np.mean(per_frame):.2f}' if per_frame else 'no detections')
    print('\nIf the clip has 1 drone, unique-IDs minus 1 = ID-switch count.')
else:
    print('No clip available — set CLIP to a video path and re-run.')

## 🖥️ C · Reusable Inference (live-demo path)

One function the live demo calls; mirrors the standalone `infer.py`.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image as IPImage, display

def run_inference(source, weights=PRIMARY_WEIGHTS, conf=0.25, save=True, name='demo_infer'):
    model = YOLO(weights)
    return model.predict(source=source, conf=conf, save=save, device=DEVICE,
                         project=RUNS_DIR_P, name=name, exist_ok=True, verbose=False)

sample_imgs = sorted(glob.glob(str(DATA_DIR_P / 'test' / 'images' / '*.jpg')))[:3]
if sample_imgs:
    res = run_inference(sample_imgs, name='demo_infer_imgs')
    for p in glob.glob(str(Path(RUNS_DIR_P) / 'demo_infer_imgs' / '*.jpg'))[:3]:
        display(IPImage(filename=p, width=520))
else:
    print("No test images found. Check DATA_DIR_P path.")

## 🎬 D · Recorded Fallback (bank this BEFORE the presentation)

Live demo is the gamble; the recorded mp4 is what gets shown if anything stalls.
Section B already produced one; this cell locates and plays it back.

In [ ]:
import glob
from IPython.display import Video, display
from pathlib import Path

d = str(Path(RUNS_DIR_P) / 'track_demo')
# prefer the re-encoded playable mp4, then any mp4, then avi
cands = (sorted(glob.glob(d + '/*_playable.mp4')) +
         sorted(glob.glob(d + '/*.mp4')) +
         sorted(glob.glob(d + '/*.avi')))
if not cands:
    print('No annotated video found — run Section B (tracking + re-encode cells) first.')
else:
    vid = cands[0]
    print('Fallback video:', vid)
    if vid.endswith('.mp4'):
        display(Video(vid, width=560, embed=True))
    else:
        print('Only .avi available — run the re-encode cell to get an inline-playable mp4.')


## 🧩 E · (Optional) SAHI: Small Distant Objects

Only for the **bird** clips with tiny distant specks; skip for drone footage (drone fills the frame).
Tiling raises small-object recall at a large FPS cost; state that tradeoff when presenting it.

In [ ]:
# SAHI tiled inference for small/distant objects.
# pip is used here (uv is not available on Colab).
!pip install -q sahi

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import cv2

_device_str = f"cuda:{DEVICE}" if isinstance(DEVICE, int) else DEVICE
det = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=PRIMARY_WEIGHTS,
    confidence_threshold=0.25,
    device=_device_str
)

bird_clips = sorted(glob.glob(str(VIDEOS_DIR / "*.mp4"))) if VIDEOS_DIR.exists() else []
SAHI_CLIP = bird_clips[-1] if bird_clips else None

if SAHI_CLIP and os.path.exists(SAHI_CLIP):
    cap = cv2.VideoCapture(SAHI_CLIP)
    cap.set(cv2.CAP_PROP_POS_FRAMES, 800)
    ok, frame = cap.read()
    cap.release()
    if ok:
        res = get_sliced_prediction(
            frame, det,
            slice_height=512, slice_width=512,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2
        )
        print("SAHI detections:", len(res.object_prediction_list))
    else:
        print("Could not read frame 800 - try a different frame number.")
else:
    print("No video clip found for SAHI. Upload .mp4 files to VIDEOS_DIR and re-run.")

## 🔁 F · Reproducibility Checklist

1. Dependencies pinned in your `.venv` (torch from the cu128 index for the RTX 5080).
2. Training used a fixed seed (`seed=42`).
3. **Before submitting**, port to Colab and do a fresh-runtime run-all there; that is the graded path.
4. Confirm `data.yaml` `path:` matches wherever the data lives in the submission environment.
5. All outputs (model weights, metrics CSV, figures) are saved to the shared Drive folder (`!CVIS/results/` and `!CVIS/models/`), so teammates can access them without re-running training.